# AG2 Beta — getting started

We'll build an agent, ask it something, then give it a tool and ask again. At the end we'll peek inside the stream and see what actually happened.

Make sure `OPENAI_API_KEY` and `EXA_API_KEY` are in your `.env`.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from autogen.beta import Agent, MemoryStream
from autogen.beta.config import OpenAIConfig

config = OpenAIConfig(
    model = "gpt-5.4-mini",
    api_key = os.getenv("OPENAI_API_KEY"),
    base_url = "https://api.openai.com/v1",
)

agent = Agent("surveyor", config=config)

## Your first ask

Here's your first agent ask. We pass in a `MemoryStream` so we can come back later and see what flowed through. The agent has no tools yet, so it'll just answer from what the model already knows.

In [3]:
from IPython.display import Markdown, display

stream = MemoryStream()
reply = await agent.ask(
    "Concisely, what are the main classes of superconductors?",
    stream=stream,
)

Markdown(reply.body)

The main classes of superconductors are:

- **Conventional superconductors**: Usually low-temperature, explained by **BCS theory** and electron-phonon pairing.
- **Unconventional superconductors**: Pairing mechanism is not the standard electron-phonon one; includes:
  - **High-temperature cuprates**
  - **Iron-based superconductors**
  - **Heavy-fermion superconductors**
  - **Organic superconductors**
  - **Some ruthenates and other exotic materials**

A common broader classification is also:

- **Type I superconductors**: Exclude magnetic fields abruptly; mostly pure elemental metals.
- **Type II superconductors**: Allow magnetic flux to penetrate in vortices; include most practical and high-field superconductors.

## Now let's add a tool

Give the agent the Exa search toolkit and ask again. We re-use the same `stream`, so the agent already remembers what it said the first time — the new question builds on that instead of starting from scratch.

In [4]:
from autogen.beta.tools.search import ExaToolkit

exa_tool = ExaToolkit(api_key = os.getenv("EXA_API_KEY"))
agent.add_tool(exa_tool)

reply = await agent.ask(
    "With exa search, concisely introduce the frontier of each class", 
    stream=stream
)

Markdown(reply.body)

Using recent reviews:

- **Conventional superconductors:** the frontier is **accurate design/prediction of phonon-mediated materials** and **engineering practical conductors** like Nb-based and HTS-related materials; ab initio Eliashberg/DFT methods are now being benchmarked for reliable \(T_c\) prediction.  
- **Unconventional superconductors:** the frontier is **understanding pairing in strongly correlated and low-dimensional systems**—especially **cuprates, iron-based systems, heavy fermions, nickelates, and BCS–BEC crossover systems**—plus disorder, strange metal behavior, and interfacial effects.  
- **Type I superconductors:** the frontier is mostly **fundamental**: clean, elemental, low-field systems are used as a baseline to study **anisotropy, disorder, and the I–II/intertype crossover**, rather than as a major application platform.  
- **Type II superconductors:** the frontier is **high-field, high-current applications**—especially **REBCO/BSCCO/MgB\(_2\)** conductors for **fusion, motors, MRI, and power devices**—with the main bottlenecks being **AC loss, quench stability, cooling cost, and manufacturing scale-up**.

## Looking inside the stream

The stream is where every event from a turn lives — your message, the model's responses, tool calls, tool results. AG2 lets you see exactly what the agent did. Below we render each event type.

In [5]:
import json
from autogen.beta.events import (
    ModelRequest, ModelResponse,
    ToolCallEvent, ToolResultsEvent,
)
from autogen.beta.tools.search.exa import (
    ExaSearchResponse, ExaSearchResult,
    ExaAnswerResult, ExaContentResult,
)

def _date(d):
    return f" · {d[:10]}" if d else ""

def _trim(s, n):
    if not s:
        return ""
    s = s.replace("\n", " ").strip()
    return s if len(s) <= n else s[:n] + "…"

def _hits(results):
    return [
        f"  {i}. [{r.title or r.url}]({r.url}){_date(r.published_date)}"
        for i, r in enumerate(results, 1)
    ]

def render_tool_data(data):
    if isinstance(data, ExaSearchResponse):
        return f"_{len(data.results)} results_\n" + "\n".join(_hits(data.results))
    if isinstance(data, ExaAnswerResult):
        body = [data.answer, "", f"_{len(data.citations)} citations_"]
        for i, c in enumerate(data.citations, 1):
            body.append(f"  {i}. [{c.title or c.url}]({c.url})")
            snippet = _trim(c.text, 120)
            if snippet:
                body.append(f"     > {snippet}")
        return "\n".join(body)
    if isinstance(data, list) and data and isinstance(data[0], ExaSearchResult):
        return "\n".join(_hits(data))
    if isinstance(data, list) and data and isinstance(data[0], ExaContentResult):
        body = [f"_{len(data)} pages_"]
        for i, c in enumerate(data, 1):
            body.append(f"  {i}. [{c.title or c.url}]({c.url})")
            snippet = _trim(c.text, 180)
            if snippet:
                body.append(f"     > {snippet}")
        return "\n".join(body)
    return f"`{data!r}`"

tool_calls: dict = {}

for ev in await stream.history.get_events():
    if isinstance(ev, ModelRequest):
        display(Markdown(f"**User** &nbsp; {ev.parts[0].content}"))

    elif isinstance(ev, ToolCallEvent):
        # Captured but not displayed — the args show up alongside the result below.
        args = json.loads(ev.arguments) if isinstance(ev.arguments, str) else ev.arguments
        tool_calls[ev.id] = {"name": ev.name, "args": args}

    elif isinstance(ev, ToolResultsEvent):
        for r in ev.results:
            call = tool_calls.get(r.parent_id, {})
            name = call.get("name", "?")
            args = call.get("args", {})
            arg_str = ", ".join(f"`{k}`={v!r}" for k, v in args.items())
            data = r.result.parts[0].data
            display(Markdown(
                f"&nbsp;&nbsp;**`{name}`** &nbsp; {arg_str}\n\n{render_tool_data(data)}"
            ))

    elif isinstance(ev, ModelResponse):
        if ev.content:
            display(Markdown(f"**Model** &nbsp; {ev.content}"))

**User** &nbsp; Concisely, what are the main classes of superconductors?

**Model** &nbsp; The main classes of superconductors are:

- **Conventional superconductors**: Usually low-temperature, explained by **BCS theory** and electron-phonon pairing.
- **Unconventional superconductors**: Pairing mechanism is not the standard electron-phonon one; includes:
  - **High-temperature cuprates**
  - **Iron-based superconductors**
  - **Heavy-fermion superconductors**
  - **Organic superconductors**
  - **Some ruthenates and other exotic materials**

A common broader classification is also:

- **Type I superconductors**: Exclude magnetic fields abruptly; mostly pure elemental metals.
- **Type II superconductors**: Allow magnetic flux to penetrate in vortices; include most practical and high-field superconductors.

**User** &nbsp; With exa search, concisely introduce the frontier of each class

&nbsp;&nbsp;**`exa_search`** &nbsp; `query`='frontier of conventional superconductors recent developments 2024 review'

_10 results_
  1. [Current Status of High Temperature Superconducting Materials and their Various Applications](https://onlinelibrary.wiley.com/doi/10.1002/tee.23976) · 2024-03-01
  2. [High-temperature superconductors and their large-scale applications | Nature Reviews Electrical Engineering](https://www.nature.com/articles/s44287-024-00112-y?error=cookies_not_supported&code=ead70738-15c5-4733-838f-843155c66f37) · 2024-11-04
  3. [Recent advances in high-entropy superconductors | NPG Asia Materials](https://www.nature.com/articles/s41427-024-00579-z) · 2024-11-29
  4. [Advancing Superconductivity with Interface Engineering (2024)](https://bishtref.com/articles/10.1002/adma.202405009) · 2024-08-06
  5. [Superconductivity and interfaces](https://www.sciencedirect.com/science/article/pii/S0370157324001558) · 2024-07-25
  6. [Hydride superconductivity is here to stay | Nature Reviews Physics](https://www.nature.com/articles/s42254-024-00794-1) · 2024-12-19
  7. [Frontiers | Editorial: Disorder and superconductivity: a 21st-century update](https://www.frontiersin.org/journals/physics/articles/10.3389/fphy.2024.1478445/full) · 2024-08-26
  8. [A review of recent advancement in superconductors - ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S2214785320375234)
  9. [When superconductivity crosses over: From BCS to BEC](https://journals.aps.org/rmp/abstract/10.1103/RevModPhys.96.025002) · 2024-05-23
  10. [CHEMICAL AND PHYSICAL FRONTIERS IN HIGH-TEMPERATURE SUPERCONDUCTIVITY
							| Kashf Journal of Multidisciplinary Research](https://kjmr.com.pk/kjmr/article/view/273) · 2025-02-12

&nbsp;&nbsp;**`exa_search`** &nbsp; `query`='frontier of unconventional superconductors cuprates iron-based heavy fermion organic recent developments 2024 review'

_10 results_
  1. [When superconductivity crosses over: From BCS to BEC](https://journals.aps.org/rmp/abstract/10.1103/RevModPhys.96.025002) · 2024-05-23
  2. [Advances in Cuprates and Iron-Based Superconductors: Physics, Properties, and Applications](https://www.mdpi.com/1996-1944/19/3/483) · 2026-01-26
  3. [Frontiers | Spontaneous time-reversal symmetry breaking by disorder in superconductors](https://www.frontiersin.org/journals/physics/articles/10.3389/fphy.2024.1353425/full) · 2024-05-20
  4. [Frontiers | Multiple superconducting phases in heavy-fermion metals](https://www.frontiersin.org/journals/electronic-materials/articles/10.3389/femat.2022.944873/full) · 2022-10-21
  5. [Editorial: New heavy fermion superconductors - Frontiers](https://www.frontiersin.org/journals/electronic-materials/articles/10.3389/femat.2022.1120381/full) · 2023-01-09
  6. [Quantum critical metals and loss of quasiparticles | Nature Physics](http://www.nature.com/articles/s41567-024-02679-7) · 2024-12-09
  7. [Charge Correlations in Cuprate Superconductors | Annual Reviews](https://www.annualreviews.org/content/journals/10.1146/annurev-conmatphys-032922-094430) · 2024-03-11
  8. [Recent advances in high-entropy superconductors | NPG Asia Materials](https://www.nature.com/articles/s41427-024-00579-z) · 2024-11-29
  9. [When Superconductivity Crosses Over: From BCS to BEC](https://arxiv.org/html/2208.01774v5)
  10. [Frontiers | Dynamic charge order from strong correlations in the cuprates](https://www.frontiersin.org/journals/electronic-materials/articles/10.3389/femat.2024.1473324/full) · 2024-11-13

&nbsp;&nbsp;**`exa_search`** &nbsp; `query`='frontier of type I superconductors recent developments 2024 review'

_10 results_
  1. [High-temperature superconductors and their large-scale applications | Nature Reviews Electrical Engineering](https://www.nature.com/articles/s44287-024-00112-y?error=cookies_not_supported&code=ead70738-15c5-4733-838f-843155c66f37) · 2024-11-04
  2. [Superconductivity and interfaces](https://www.sciencedirect.com/science/article/pii/S0370157324001558) · 2024-07-25
  3. [Intertype superconductivity evoked by the interplay of disorder and multiple bands | Frontiers of Physics | Springer Nature Link](https://link.springer.com/article/10.1007/s11467-023-1379-y) · 2024-01-24
  4. [A review of recent advancement in superconductors - ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S2214785320375234)
  5. [Frontiers | Effects of anisotropy and disorder on the superconducting properties of niobium](https://www.frontiersin.org/articles/10.3389/fphy.2023.1269872/full) · 2023-10-30
  6. [Ab initio methods for superconductivity | Nature Reviews Physics](http://www.nature.com/articles/s42254-024-00738-9) · 2024-07-29
  7. [Advancing Superconductivity with Interface Engineering (2024)](https://bishtref.com/articles/10.1002/adma.202405009) · 2024-08-06
  8. [Frontiers | Spontaneous time-reversal symmetry breaking by disorder in superconductors](https://www.frontiersin.org/journals/physics/articles/10.3389/fphy.2024.1353425/full) · 2024-05-20
  9. [Editorial: Disorder and superconductivity: a 21st-century update](https://www.frontiersin.org/journals/physics/articles/10.3389/fphy.2024.1478445/full) · 2024-08-26
  10. [Disorder and Superconductivity: a 21st-century update | Frontiers Research Topic](https://www.frontiersin.org/research-topics/55749/disorder-and-superconductivity-a-21st-century-update) · 2023-06-09

&nbsp;&nbsp;**`exa_search`** &nbsp; `query`='frontier of type II superconductors recent developments 2024 review'

_10 results_
  1. [High-temperature superconductors and their large-scale applications | Nature Reviews Electrical Engineering](https://www.nature.com/articles/s44287-024-00112-y?error=cookies_not_supported&code=ead70738-15c5-4733-838f-843155c66f37) · 2024-11-04
  2. [Superconductivity and interfaces](https://www.sciencedirect.com/science/article/pii/S0370157324001558) · 2024-07-25
  3. [Advancing Superconductivity with Interface Engineering (2024)](https://bishtref.com/articles/10.1002/adma.202405009) · 2024-08-06
  4. [Frontiers | Editorial: Disorder and superconductivity: a 21st-century update](https://www.frontiersin.org/journals/physics/articles/10.3389/fphy.2024.1478445/full) · 2024-08-26
  5. [Ab initio methods for superconductivity | Nature Reviews Physics](http://www.nature.com/articles/s42254-024-00738-9) · 2024-07-29
  6. [A review of recent advancement in superconductors - ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S2214785320375234)
  7. [CHEMICAL AND PHYSICAL FRONTIERS IN HIGH-TEMPERATURE SUPERCONDUCTIVITY
							| Kashf Journal of Multidisciplinary Research](https://kjmr.com.pk/kjmr/article/view/273) · 2025-02-12
  8. [Recent progress in nickelate superconductors - Oxford Academic](https://academic.oup.com/nsr/article/12/10/nwaf373/8248521) · 2025-09-09
  9. [New study reviews progress in ternary hydrogen-rich superconductors](https://phys.org/news/2024-08-ternary-hydrogen-rich-superconductors.html) · 2024-08-01
  10. [Type I & II Superconductors | Applications, Theory & Temp Limits](https://modern-physics.org/type-i-and-type-ii-superconductors/) · 2024-05-28

**Model** &nbsp; Using recent reviews:

- **Conventional superconductors:** the frontier is **accurate design/prediction of phonon-mediated materials** and **engineering practical conductors** like Nb-based and HTS-related materials; ab initio Eliashberg/DFT methods are now being benchmarked for reliable \(T_c\) prediction.  
- **Unconventional superconductors:** the frontier is **understanding pairing in strongly correlated and low-dimensional systems**—especially **cuprates, iron-based systems, heavy fermions, nickelates, and BCS–BEC crossover systems**—plus disorder, strange metal behavior, and interfacial effects.  
- **Type I superconductors:** the frontier is mostly **fundamental**: clean, elemental, low-field systems are used as a baseline to study **anisotropy, disorder, and the I–II/intertype crossover**, rather than as a major application platform.  
- **Type II superconductors:** the frontier is **high-field, high-current applications**—especially **REBCO/BSCCO/MgB\(_2\)** conductors for **fusion, motors, MRI, and power devices**—with the main bottlenecks being **AC loss, quench stability, cooling cost, and manufacturing scale-up**.